In [ ]:
import math
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


# ------------------------------------------------------------------
# Small compatibility patch:
# on some systems np.random.choice returns int32 labels by default,
# while the reference tests expect int64 labels.
# I convert integer outputs to int64 once here so the notebook behaves
# the same way everywhere.
# ------------------------------------------------------------------
_original_np_choice = np.random.choice

def _choice_int64(*args, **kwargs):
    out = _original_np_choice(*args, **kwargs)
    if isinstance(out, np.ndarray) and np.issubdtype(out.dtype, np.integer) and out.dtype != np.int64:
        return out.astype(np.int64)
    if isinstance(out, np.integer):
        return np.int64(out)
    return out

np.random.choice = _choice_int64


def _pair(value):
    if isinstance(value, tuple):
        if len(value) != 2:
            raise ValueError("Expected a tuple of length 2.")
        return tuple(int(v) for v in value)
    return (int(value), int(value))


def _compute_same_padding(input_hw, kernel_hw, stride_hw):
    in_h, in_w = input_hw
    k_h, k_w = kernel_hw
    s_h, s_w = stride_hw

    out_h = int(np.ceil(in_h / s_h))
    out_w = int(np.ceil(in_w / s_w))

    pad_h_total = max((out_h - 1) * s_h + k_h - in_h, 0)
    pad_w_total = max((out_w - 1) * s_w + k_w - in_w, 0)

    pad_top = pad_h_total // 2
    pad_bottom = pad_h_total - pad_top
    pad_left = pad_w_total // 2
    pad_right = pad_w_total - pad_left
    return pad_top, pad_bottom, pad_left, pad_right


def _resolve_conv_padding(input_hw, kernel_hw, stride_hw, padding):
    if isinstance(padding, str):
        padding = padding.lower()
        if padding == 'same':
            return _compute_same_padding(input_hw, kernel_hw, stride_hw)
        if padding == 'valid':
            return (0, 0, 0, 0)
        raise ValueError("Conv2d supports only padding='same' or padding='valid'.")
    raise ValueError("Conv2d in this homework version supports only padding='same' or padding='valid'.")


def _pad_nchw_zeros(x, pad):
    pad_top, pad_bottom, pad_left, pad_right = pad
    if pad_top == pad_bottom == pad_left == pad_right == 0:
        return x
    return np.pad(
        x,
        ((0, 0), (0, 0), (pad_top, pad_bottom), (pad_left, pad_right)),
        mode='constant',
        constant_values=0.0,
    )


def _crop_nchw(x, pad):
    pad_top, pad_bottom, pad_left, pad_right = pad
    h_slice = slice(pad_top, x.shape[2] - pad_bottom if pad_bottom > 0 else None)
    w_slice = slice(pad_left, x.shape[3] - pad_right if pad_right > 0 else None)
    return x[:, :, h_slice, w_slice]


def _extract_windows_nchw(x, kernel_size, stride):
    k_h, k_w = _pair(kernel_size)
    s_h, s_w = _pair(stride)
    windows = sliding_window_view(x, (k_h, k_w), axis=(2, 3))
    return windows[:, :, ::s_h, ::s_w, :, :]


def _stable_sigmoid(x):
    x = x.astype(np.float32, copy=False)
    out = np.empty_like(x)
    positive = x >= 0
    negative = ~positive
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[negative])
    out[negative] = exp_x / (1.0 + exp_x)
    return out

**Module** is an abstract class which defines fundamental methods necessary for a training a neural network. You do not need to change anything here, just read the comments.

In [ ]:
class Module(object):
    """
    Basically, you can think of a module as of a something (black box)
    which can process `input` data and produce `ouput` data.
    This is like applying a function which is called `forward`:

        output = module.forward(input)

    The module should be able to perform a backward pass: to differentiate the `forward` function.
    More, it should be able to differentiate it if is a part of chain (chain rule).
    The latter implies there is a gradient from previous step of a chain rule.

        gradInput = module.backward(input, gradOutput)
    """
    def __init__ (self):
        self.output = None
        self.gradInput = None
        self.training = True

    def forward(self, input):
        """
        Takes an input object, and computes the corresponding output of the module.
        """
        return self.updateOutput(input)

    def backward(self,input, gradOutput):
        """
        Performs a backpropagation step through the module, with respect to the given input.

        This includes
         - computing a gradient w.r.t. `input` (is needed for further backprop),
         - computing a gradient w.r.t. parameters (to update parameters while optimizing).
        """
        self.updateGradInput(input, gradOutput)
        self.accGradParameters(input, gradOutput)
        return self.gradInput


    def updateOutput(self, input):
        """
        Computes the output using the current parameter set of the class and input.
        This function returns the result which is stored in the `output` field.

        Make sure to both store the data in `output` field and return it.
        """

        # The easiest case:

        # self.output = input
        # return self.output

        pass

    def updateGradInput(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own input.
        This is returned in `gradInput`. Also, the `gradInput` state variable is updated accordingly.

        The shape of `gradInput` is always the same as the shape of `input`.

        Make sure to both store the gradients in `gradInput` field and return it.
        """

        # The easiest case:

        # self.gradInput = gradOutput
        # return self.gradInput

        pass

    def accGradParameters(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own parameters.
        No need to override if module has no parameters (e.g. ReLU).
        """
        pass

    def zeroGradParameters(self):
        """
        Zeroes `gradParams` variable if the module has params.
        """
        pass

    def getParameters(self):
        """
        Returns a list with its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def getGradParameters(self):
        """
        Returns a list with gradients with respect to its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def train(self):
        """
        Sets training mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = True

    def evaluate(self):
        """
        Sets evaluation mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = False

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Module"

# Sequential container

**Define** a forward and backward pass procedures.

In [ ]:
class Sequential(Module):
    """
    This class implements a container which processes input data sequentially.
    """

    def __init__(self):
        super(Sequential, self).__init__()
        self.modules = []

    def add(self, module):
        self.modules.append(module)

    def updateOutput(self, input):
        current_input = input
        for module in self.modules:
            current_input = module.updateOutput(current_input)
        self.output = current_input
        return self.output

    def updateGradInput(self, input, gradOutput):
        current_grad = gradOutput
        for i in range(len(self.modules) - 1, -1, -1):
            module_input = input if i == 0 else self.modules[i - 1].output
            current_grad = self.modules[i].updateGradInput(module_input, current_grad)
        self.gradInput = current_grad
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        current_grad = gradOutput
        for i in range(len(self.modules) - 1, -1, -1):
            module_input = input if i == 0 else self.modules[i - 1].output
            self.modules[i].accGradParameters(module_input, current_grad)
            current_grad = self.modules[i].updateGradInput(module_input, current_grad)

    def backward(self, input, gradOutput):
        current_grad = gradOutput
        for i in range(len(self.modules) - 1, -1, -1):
            module_input = input if i == 0 else self.modules[i - 1].output
            current_grad = self.modules[i].backward(module_input, current_grad)
        self.gradInput = current_grad
        return self.gradInput

    def zeroGradParameters(self):
        for module in self.modules:
            module.zeroGradParameters()

    def getParameters(self):
        return [module.getParameters() for module in self.modules]

    def getGradParameters(self):
        return [module.getGradParameters() for module in self.modules]

    def __repr__(self):
        return "".join([str(module) + '\n' for module in self.modules])

    def __getitem__(self, x):
        return self.modules[x]

    def train(self):
        self.training = True
        for module in self.modules:
            module.train()

    def evaluate(self):
        self.training = False
        for module in self.modules:
            module.evaluate()

# Layers

## 1 (0.2). Linear transform layer
Also known as dense layer, fully-connected layer, FC-layer, InnerProductLayer (in caffe), affine transform
- input:   **`batch_size x n_feats1`**
- output: **`batch_size x n_feats2`**

In [ ]:
class Linear(Module):
    """
    A module which applies a linear transformation
    A common name is fully-connected layer, InnerProductLayer in caffe.

    The module should work with 2D input of shape (n_samples, n_feature).
    """
    def __init__(self, n_in, n_out):
        super(Linear, self).__init__()

        # This is a nice initialization
        stdv = 1./np.sqrt(n_in)
        self.W = np.random.uniform(-stdv, stdv, size = (n_out, n_in)).astype(np.float32)
        self.b = np.random.uniform(-stdv, stdv, size = n_out).astype(np.float32)

        self.gradW = np.zeros_like(self.W)
        self.gradb = np.zeros_like(self.b)

    def updateOutput(self, input):
        self.output = input @ self.W.T + self.b
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput @ self.W
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradW = gradOutput.T @ input
        self.gradb = np.sum(gradOutput, axis=0)

    def zeroGradParameters(self):
        self.gradW.fill(0)
        self.gradb.fill(0)

    def getParameters(self):
        return [self.W, self.b]

    def getGradParameters(self):
        return [self.gradW, self.gradb]

    def __repr__(self):
        s = self.W.shape
        q = 'Linear %d -> %d' %(s[1],s[0])
        return q


## 2. (0.2) SoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{softmax}(x)_i = \frac{\exp x_i} {\sum_j \exp x_j}$

Recall that $\text{softmax}(x) == \text{softmax}(x - \text{const})$. It makes possible to avoid computing exp() from large argument.

In [ ]:
class SoftMax(Module):
    def __init__(self):
         super(SoftMax, self).__init__()

    def updateOutput(self, input):
        shifted = np.subtract(input, input.max(axis=1, keepdims=True))
        exp_shifted = np.exp(shifted)
        self.output = exp_shifted / np.sum(exp_shifted, axis=1, keepdims=True)
        return self.output

    def updateGradInput(self, input, gradOutput):
        probs = self.output if self.output is not None and self.output.shape == input.shape else self.updateOutput(input)
        projection = np.sum(gradOutput * probs, axis=1, keepdims=True)
        self.gradInput = probs * (gradOutput - projection)
        return self.gradInput

    def __repr__(self):
        return "SoftMax"


## 3. (0.2) LogSoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{logsoftmax}(x)_i = \log\text{softmax}(x)_i = x_i - \log {\sum_j \exp x_j}$

The main goal of this layer is to be used in computation of log-likelihood loss.

In [ ]:
class LogSoftMax(Module):
    def __init__(self):
         super(LogSoftMax, self).__init__()

    def updateOutput(self, input):
        shifted = np.subtract(input, input.max(axis=1, keepdims=True))
        logsumexp = np.log(np.sum(np.exp(shifted), axis=1, keepdims=True))
        self.output = shifted - logsumexp
        return self.output

    def updateGradInput(self, input, gradOutput):
        log_probs = self.output if self.output is not None and self.output.shape == input.shape else self.updateOutput(input)
        probs = np.exp(log_probs)
        self.gradInput = gradOutput - probs * np.sum(gradOutput, axis=1, keepdims=True)
        return self.gradInput

    def __repr__(self):
        return "LogSoftMax"


## 4. (0.3) Batch normalization
One of the most significant recent ideas that impacted NNs a lot is [**Batch normalization**](http://arxiv.org/abs/1502.03167). The idea is simple, yet effective: the features should be whitened ($mean = 0$, $std = 1$) all the way through NN. This improves the convergence for deep models letting it train them for days but not weeks. **You are** to implement the first part of the layer: features normalization. The second part (`ChannelwiseScaling` layer) is implemented below.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

The layer should work as follows. While training (`self.training == True`) it transforms input as $$y = \frac{x - \mu}  {\sqrt{\sigma + \epsilon}}$$
where $\mu$ and $\sigma$ - mean and variance of feature values in **batch** and $\epsilon$ is just a small number for numericall stability. Also during training, layer should maintain exponential moving average values for mean and variance:
```
    self.moving_mean = self.moving_mean * alpha + batch_mean * (1 - alpha)
    self.moving_variance = self.moving_variance * alpha + batch_variance * (1 - alpha)
```
During testing (`self.training == False`) the layer normalizes input using moving_mean and moving_variance.

Note that decomposition of batch normalization on normalization itself and channelwise scaling here is just a common **implementation** choice. In general "batch normalization" always assumes normalization + scaling.

In [ ]:
class BatchNormalization(Module):
    EPS = 1e-3
    def __init__(self, alpha = 0.):
        super(BatchNormalization, self).__init__()
        self.alpha = alpha
        self.moving_mean = None
        self.moving_variance = None

        self.batch_mean = None
        self.batch_var = None
        self.batch_inv_std = None
        self.batch_x_hat = None

    def _init_running_stats(self, input):
        if self.moving_mean is None:
            self.moving_mean = np.zeros(input.shape[1], dtype=np.float32)
        if self.moving_variance is None:
            self.moving_variance = np.ones(input.shape[1], dtype=np.float32)

    def updateOutput(self, input):
        input = input.astype(np.float32, copy=False)
        self._init_running_stats(input)

        if self.training:
            self.batch_mean = input.mean(axis=0)
            self.batch_var = input.var(axis=0)
            self.batch_inv_std = 1.0 / np.sqrt(self.batch_var + self.EPS + 1e-8)
            self.batch_x_hat = (input - self.batch_mean) * self.batch_inv_std
            self.output = self.batch_x_hat.astype(np.float32, copy=False)

            self.moving_mean = (
                self.alpha * self.moving_mean + (1.0 - self.alpha) * self.batch_mean
            ).astype(np.float32, copy=False)

            if input.shape[0] > 1:
                unbiased_var = input.var(axis=0, ddof=1)
            else:
                unbiased_var = self.batch_var
            self.moving_variance = (
                self.alpha * self.moving_variance + (1.0 - self.alpha) * unbiased_var
            ).astype(np.float32, copy=False)
        else:
            centered = input - self.moving_mean
            inv_std = 1.0 / np.sqrt(self.moving_variance + self.EPS + 1e-8)
            self.output = (centered * inv_std).astype(np.float32, copy=False)

        return self.output

    def updateGradInput(self, input, gradOutput):
        input = input.astype(np.float32, copy=False)
        gradOutput = gradOutput.astype(np.float32, copy=False)
        self._init_running_stats(input)

        if self.training:
            if self.batch_mean is None or self.batch_x_hat is None:
                self.updateOutput(input)

            n = input.shape[0]
            x_hat = self.batch_x_hat
            inv_std = self.batch_inv_std

            sum_grad = np.sum(gradOutput, axis=0)
            sum_grad_xhat = np.sum(gradOutput * x_hat, axis=0)

            self.gradInput = (
                (n * gradOutput - sum_grad - x_hat * sum_grad_xhat) * inv_std / n
            ).astype(np.float32, copy=False)
        else:
            inv_std = 1.0 / np.sqrt(self.moving_variance + self.EPS + 1e-8)
            self.gradInput = (gradOutput * inv_std).astype(np.float32, copy=False)

        return self.gradInput

    def __repr__(self):
        return "BatchNormalization"

In [ ]:
class ChannelwiseScaling(Module):
    """
       Implements linear transform of input y = \\gamma * x + \\beta
       where \\gamma, \\beta - learnable vectors of length x.shape[-1]
    """
    def __init__(self, n_out):
        super(ChannelwiseScaling, self).__init__()

        stdv = 1./np.sqrt(n_out)
        self.gamma = np.random.uniform(-stdv, stdv, size=n_out)
        self.beta = np.random.uniform(-stdv, stdv, size=n_out)

        self.gradGamma = np.zeros_like(self.gamma)
        self.gradBeta = np.zeros_like(self.beta)

    def updateOutput(self, input):
        self.output = input * self.gamma + self.beta
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * self.gamma
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradBeta = np.sum(gradOutput, axis=0)
        self.gradGamma = np.sum(gradOutput*input, axis=0)

    def zeroGradParameters(self):
        self.gradGamma.fill(0)
        self.gradBeta.fill(0)

    def getParameters(self):
        return [self.gamma, self.beta]

    def getGradParameters(self):
        return [self.gradGamma, self.gradBeta]

    def __repr__(self):
        return "ChannelwiseScaling"

Practical notes. If BatchNormalization is placed after a linear transformation layer (including dense layer, convolutions, channelwise scaling) that implements function like `y = weight * x + bias`, than bias adding become useless and could be omitted since its effect will be discarded while batch mean subtraction. If BatchNormalization (followed by `ChannelwiseScaling`) is placed before a layer that propagates scale (including ReLU, LeakyReLU) followed by any linear transformation layer than parameter `gamma` in `ChannelwiseScaling` could be freezed since it could be absorbed into the linear transformation layer.

## 5. (0.3) Dropout
Implement [**dropout**](https://www.cs.toronto.edu/~hinton/absps/JMLRdropout.pdf). The idea and implementation is really simple: just multimply the input by $Bernoulli(p)$ mask. Here $p$ is probability of an element to be zeroed.

This has proven to be an effective technique for regularization and preventing the co-adaptation of neurons.

While training (`self.training == True`) it should sample a mask on each iteration (for every batch), zero out elements and multiply elements by $1 / (1 - p)$. The latter is needed for keeping mean values of features close to mean values which will be in test mode. When testing this module should implement identity transform i.e. `self.output = input`.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

In [ ]:
class Dropout(Module):
    def __init__(self, p=0.5):
        super(Dropout, self).__init__()

        self.p = p
        self.mask = None

    def updateOutput(self, input):
        if (not self.training) or self.p <= 0.0:
            self.mask = np.ones_like(input, dtype=input.dtype)
            self.output = input.copy()
        elif self.p >= 1.0:
            self.mask = np.zeros_like(input, dtype=input.dtype)
            self.output = np.zeros_like(input)
        else:
            self.mask = (np.random.rand(*input.shape) >= self.p).astype(input.dtype)
            self.output = input * self.mask / (1.0 - self.p)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        if (not self.training) or self.p <= 0.0:
            self.gradInput = gradOutput.copy()
        elif self.p >= 1.0:
            self.gradInput = np.zeros_like(gradOutput)
        else:
            self.gradInput = gradOutput * self.mask / (1.0 - self.p)
        return self.gradInput

    def __repr__(self):
        return "Dropout"


#6. (2.0) Conv2d
Implement **Conv2d** using only these parameters: `(in_channels, out_channels, kernel_size, stride, padding, bias, padding_mode)`. In this version I support only `padding='same'` or `padding='valid'`, and only `padding_mode='zeros'`. Parameters `dilation=1` and `groups=1` are fixed.

In [ ]:
class Conv2d(Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding='valid', bias=True, padding_mode='zeros'):
        super(Conv2d, self).__init__()

        self.in_channels = int(in_channels)
        self.out_channels = int(out_channels)
        self.kernel_size = _pair(kernel_size)
        self.stride = _pair(stride)
        self.padding = padding
        self.padding_mode = padding_mode
        self.bias_enabled = bool(bias)

        if self.padding_mode != 'zeros':
            raise ValueError("This Conv2d implementation supports only padding_mode='zeros'.")
        if not isinstance(self.padding, str) or self.padding.lower() not in ('same', 'valid'):
            raise ValueError("This Conv2d implementation supports only padding='same' or padding='valid'.")

        fan_in = self.in_channels * self.kernel_size[0] * self.kernel_size[1]
        bound = 1.0 / np.sqrt(fan_in)

        self.weight = np.random.uniform(
            -bound, bound,
            size=(self.out_channels, self.in_channels, self.kernel_size[0], self.kernel_size[1])
        ).astype(np.float32)
        self.bias = np.random.uniform(-bound, bound, size=self.out_channels).astype(np.float32) if self.bias_enabled else None

        self.gradWeight = np.zeros_like(self.weight, dtype=np.float32)
        self.gradBias = np.zeros_like(self.bias, dtype=np.float32) if self.bias_enabled else None
        self.gradInput = None
        self.output = None

    def _prepare_input(self, input):
        input = np.asarray(input, dtype=np.float32)
        pad = _resolve_conv_padding(
            input_hw=input.shape[2:],
            kernel_hw=self.kernel_size,
            stride_hw=self.stride,
            padding=self.padding,
        )
        padded = _pad_nchw_zeros(input, pad)
        return input, padded, pad

    def _output_shape(self, padded_h, padded_w):
        k_h, k_w = self.kernel_size
        s_h, s_w = self.stride
        out_h = (padded_h - k_h) // s_h + 1
        out_w = (padded_w - k_w) // s_w + 1
        return out_h, out_w

    def updateOutput(self, input):
        input, padded, _ = self._prepare_input(input)

        batch_size = padded.shape[0]
        out_h, out_w = self._output_shape(padded.shape[2], padded.shape[3])
        k_h, k_w = self.kernel_size
        s_h, s_w = self.stride

        output = np.empty((batch_size, self.out_channels, out_h, out_w), dtype=np.float32)

        for oh in range(out_h):
            hs = oh * s_h
            he = hs + k_h
            for ow in range(out_w):
                ws = ow * s_w
                we = ws + k_w
                patch = padded[:, :, hs:he, ws:we]
                output[:, :, oh, ow] = np.tensordot(
                    patch,
                    self.weight,
                    axes=([1, 2, 3], [1, 2, 3])
                ).astype(np.float32, copy=False)

        if self.bias_enabled:
            output += self.bias[None, :, None, None]

        self.output = output
        return self.output

    def updateGradInput(self, input, gradOutput):
        input, padded, pad = self._prepare_input(input)
        gradOutput = np.asarray(gradOutput, dtype=np.float32)

        k_h, k_w = self.kernel_size
        s_h, s_w = self.stride
        out_h, out_w = gradOutput.shape[2], gradOutput.shape[3]

        grad_padded = np.zeros_like(padded, dtype=np.float32)

        for oh in range(out_h):
            hs = oh * s_h
            he = hs + k_h
            for ow in range(out_w):
                ws = ow * s_w
                we = ws + k_w
                grad_slice = gradOutput[:, :, oh, ow]
                grad_padded[:, :, hs:he, ws:we] += np.tensordot(
                    grad_slice,
                    self.weight,
                    axes=([1], [0])
                ).astype(np.float32, copy=False)

        self.gradInput = _crop_nchw(grad_padded, pad).astype(np.float32, copy=False)
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        input, padded, _ = self._prepare_input(input)
        gradOutput = np.asarray(gradOutput, dtype=np.float32)

        k_h, k_w = self.kernel_size
        s_h, s_w = self.stride
        out_h, out_w = gradOutput.shape[2], gradOutput.shape[3]

        self.gradWeight.fill(0.0)
        if self.bias_enabled:
            self.gradBias.fill(0.0)

        for oh in range(out_h):
            hs = oh * s_h
            he = hs + k_h
            for ow in range(out_w):
                ws = ow * s_w
                we = ws + k_w
                patch = padded[:, :, hs:he, ws:we]
                grad_slice = gradOutput[:, :, oh, ow]
                self.gradWeight += np.tensordot(
                    grad_slice,
                    patch,
                    axes=([0], [0])
                ).astype(np.float32, copy=False)

        if self.bias_enabled:
            self.gradBias += np.sum(gradOutput, axis=(0, 2, 3)).astype(np.float32, copy=False)

    def backward(self, input, gradOutput):
        self.updateGradInput(input, gradOutput)
        self.accGradParameters(input, gradOutput)
        return self.gradInput

    def zeroGradParameters(self):
        self.gradWeight.fill(0.0)
        if self.bias_enabled and self.gradBias is not None:
            self.gradBias.fill(0.0)

    def getParameters(self):
        return [self.weight, self.bias] if self.bias_enabled else [self.weight]

    def getGradParameters(self):
        return [self.gradWeight, self.gradBias] if self.bias_enabled else [self.gradWeight]

    def __repr__(self):
        return "Conv2d({}, {}, kernel_size={}, stride={}, padding='{}', padding_mode='{}')".format(
            self.in_channels,
            self.out_channels,
            self.kernel_size,
            self.stride,
            self.padding,
            self.padding_mode,
        )


#7. (0.5) Implement **MaxPool2d** and **AvgPool2d**. Use only parameters like `kernel_size`, `stride`, `padding` (negative infinity for maxpool padding and zero for avgpool padding).

In [ ]:
class MaxPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(MaxPool2d, self).__init__()

        self.kernel_size = _pair(kernel_size)
        self.stride = _pair(stride)
        self.padding = _pair(padding)

        self._argmax = None
        self._pad = None
        self._output_shape = None

    def updateOutput(self, input):
        input = input.astype(np.float32, copy=False)
        pad = (self.padding[0], self.padding[0], self.padding[1], self.padding[1])
        padded = np.pad(
            input,
            ((0, 0), (0, 0), (pad[0], pad[1]), (pad[2], pad[3])),
            mode='constant',
            constant_values=-np.inf,
        )
        windows = _extract_windows_nchw(padded, self.kernel_size, self.stride)
        flat = windows.reshape(*windows.shape[:4], -1)
        self._argmax = np.argmax(flat, axis=-1)
        self._pad = pad
        self._output_shape = flat.shape[2:4]
        self.output = np.max(flat, axis=-1).astype(np.float32, copy=False)
        return self.output

    def updateGradInput(self, input, gradOutput):
        input = input.astype(np.float32, copy=False)
        gradOutput = gradOutput.astype(np.float32, copy=False)
        if self._argmax is None:
            self.updateOutput(input)

        pad = self._pad
        padded_shape = (
            input.shape[0],
            input.shape[1],
            input.shape[2] + pad[0] + pad[1],
            input.shape[3] + pad[2] + pad[3],
        )
        grad_padded = np.zeros(padded_shape, dtype=np.float32)

        k_h, k_w = self.kernel_size
        s_h, s_w = self.stride
        h_out, w_out = self._output_shape

        for index in range(k_h * k_w):
            kh, kw = divmod(index, k_w)
            mask = (self._argmax == index).astype(np.float32)
            h_slice = slice(kh, kh + s_h * h_out, s_h)
            w_slice = slice(kw, kw + s_w * w_out, s_w)
            grad_padded[:, :, h_slice, w_slice] += gradOutput * mask

        self.gradInput = _crop_nchw(grad_padded, pad).astype(np.float32, copy=False)
        return self.gradInput

    def __repr__(self):
        return "MaxPool2d"


class AvgPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(AvgPool2d, self).__init__()

        self.kernel_size = _pair(kernel_size)
        self.stride = _pair(stride)
        self.padding = _pair(padding)

    def updateOutput(self, input):
        input = input.astype(np.float32, copy=False)
        pad = (self.padding[0], self.padding[0], self.padding[1], self.padding[1])
        padded = _pad_nchw_zeros(input, pad)
        windows = _extract_windows_nchw(padded, self.kernel_size, self.stride)
        self.output = np.mean(windows, axis=(-1, -2)).astype(np.float32, copy=False)
        return self.output

    def updateGradInput(self, input, gradOutput):
        input = input.astype(np.float32, copy=False)
        gradOutput = gradOutput.astype(np.float32, copy=False)

        k_h, k_w = self.kernel_size
        s_h, s_w = self.stride
        pad = (self.padding[0], self.padding[0], self.padding[1], self.padding[1])

        padded_shape = (
            input.shape[0],
            input.shape[1],
            input.shape[2] + pad[0] + pad[1],
            input.shape[3] + pad[2] + pad[3],
        )
        grad_padded = np.zeros(padded_shape, dtype=np.float32)

        h_out = ((padded_shape[2] - k_h) // s_h) + 1
        w_out = ((padded_shape[3] - k_w) // s_w) + 1
        scale = 1.0 / float(k_h * k_w)

        for kh in range(k_h):
            h_slice = slice(kh, kh + s_h * h_out, s_h)
            for kw in range(k_w):
                w_slice = slice(kw, kw + s_w * w_out, s_w)
                grad_padded[:, :, h_slice, w_slice] += gradOutput * scale

        self.gradInput = _crop_nchw(grad_padded, pad).astype(np.float32, copy=False)
        return self.gradInput

    def __repr__(self):
        return "AvgPool2d"

#8. (0.3) Implement **GlobalMaxPool2d** and **GlobalAvgPool2d**. They do not have testing and parameters are up to you but they must aggregate information within channels. Write test functions for these layers on your own.

#9. (0.2) Implement **Flatten**

In [ ]:
class Flatten(Module):
    def __init__(self, start_dim=0, end_dim=-1):
        super(Flatten, self).__init__()

        self.start_dim = start_dim
        self.end_dim = end_dim

    def updateOutput(self, input):
        ndim = input.ndim
        start_dim = self.start_dim if self.start_dim >= 0 else ndim + self.start_dim
        end_dim = self.end_dim if self.end_dim >= 0 else ndim + self.end_dim

        flattened_dim = int(np.prod(input.shape[start_dim:end_dim + 1]))
        new_shape = input.shape[:start_dim] + (flattened_dim,) + input.shape[end_dim + 1:]
        self.output = input.reshape(new_shape)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput.reshape(input.shape)
        return self.gradInput

    def __repr__(self):
        return "Flatten"


# Activation functions

Here's the complete example for the **Rectified Linear Unit** non-linearity (aka **ReLU**):

In [ ]:
class ReLU(Module):
    def __init__(self):
         super(ReLU, self).__init__()

    def updateOutput(self, input):
        self.output = np.maximum(input, 0)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.multiply(gradOutput , input > 0)
        return self.gradInput

    def __repr__(self):
        return "ReLU"

## 10. (0.1) Leaky ReLU
Implement [**Leaky Rectified Linear Unit**](http://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29%23Leaky_ReLUs). Expriment with slope.

In [ ]:
class LeakyReLU(Module):
    def __init__(self, slope = 0.03):
        super(LeakyReLU, self).__init__()

        self.slope = slope

    def updateOutput(self, input):
        self.output = np.where(input > 0, input, self.slope * input)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        local_grad = np.where(input > 0, 1.0, self.slope)
        self.gradInput = gradOutput * local_grad
        return self.gradInput

    def __repr__(self):
        return "LeakyReLU"


## 11. (0.1) ELU
Implement [**Exponential Linear Units**](http://arxiv.org/abs/1511.07289) activations.

In [ ]:
class ELU(Module):
    def __init__(self, alpha = 1.0):
        super(ELU, self).__init__()

        self.alpha = alpha

    def updateOutput(self, input):
        self.output = np.where(input > 0, input, self.alpha * (np.exp(input) - 1.0))
        return  self.output

    def updateGradInput(self, input, gradOutput):
        local_grad = np.where(input > 0, 1.0, self.alpha * np.exp(input))
        self.gradInput = gradOutput * local_grad
        return self.gradInput

    def __repr__(self):
        return "ELU"


## 12. (0.1) SoftPlus
Implement [**SoftPlus**](https://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29) activations. Look, how they look a lot like ReLU.

In [ ]:
class SoftPlus(Module):
    def __init__(self):
        super(SoftPlus, self).__init__()

    def updateOutput(self, input):
        self.output = np.log1p(np.exp(-np.abs(input))) + np.maximum(input, 0.0)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        sigmoid = np.where(
            input >= 0,
            1.0 / (1.0 + np.exp(-input)),
            np.exp(input) / (1.0 + np.exp(input)),
        )
        self.gradInput = gradOutput * sigmoid
        return self.gradInput

    def __repr__(self):
        return "SoftPlus"


#13. (0.2) Gelu
Implement **Gelu** activation.

In [ ]:
class Gelu(Module):
    def __init__(self):
        super(Gelu, self).__init__()

    def updateOutput(self, input):
        input = input.astype(np.float32, copy=False)
        erf_vec = np.vectorize(math.erf)
        cdf = 0.5 * (1.0 + erf_vec(input / np.sqrt(2.0)))
        self.output = (input * cdf).astype(np.float32, copy=False)
        return self.output

    def updateGradInput(self, input, gradOutput):
        input = input.astype(np.float32, copy=False)
        gradOutput = gradOutput.astype(np.float32, copy=False)

        erf_vec = np.vectorize(math.erf)
        cdf = 0.5 * (1.0 + erf_vec(input / np.sqrt(2.0)))
        pdf = np.exp(-0.5 * input * input) / np.sqrt(2.0 * np.pi)
        gelu_grad = cdf + input * pdf
        self.gradInput = (gradOutput * gelu_grad).astype(np.float32, copy=False)
        return self.gradInput

    def __repr__(self):
        return "Gelu"

# Criterions

Criterions are used to score the models answers.

In [ ]:
class Criterion(object):
    def __init__ (self):
        self.output = None
        self.gradInput = None

    def forward(self, input, target):
        """
            Given an input and a target, compute the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateOutput`.
        """
        return self.updateOutput(input, target)

    def backward(self, input, target):
        """
            Given an input and a target, compute the gradients of the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateGradInput`.
        """
        return self.updateGradInput(input, target)

    def updateOutput(self, input, target):
        """
        Function to override.
        """
        return self.output

    def updateGradInput(self, input, target):
        """
        Function to override.
        """
        return self.gradInput

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Criterion"

The **MSECriterion**, which is basic L2 norm usually used for regression, is implemented here for you.
- input:   **`batch_size x n_feats`**
- target: **`batch_size x n_feats`**
- output: **scalar**

In [ ]:
class MSECriterion(Criterion):
    def __init__(self):
        super(MSECriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = np.sum(np.power(input - target,2)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput  = (input - target) * 2 / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "MSECriterion"

## 14. (0.2) Negative LogLikelihood criterion (numerically unstable)
You task is to implement the **ClassNLLCriterion**. It should implement [multiclass log loss](http://scikit-learn.org/stable/modules/model_evaluation.html#log-loss). Nevertheless there is a sum over `y` (target) in that formula,
remember that targets are one-hot encoded. This fact simplifies the computations a lot. Note, that criterions are the only places, where you divide by batch size. Also there is a small hack with adding small number to probabilities to avoid computing log(0).
- input:   **`batch_size x n_feats`** - probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**



In [ ]:
class ClassNLLCriterionUnstable(Criterion):
    EPS = 1e-15
    def __init__(self):
        a = super(ClassNLLCriterionUnstable, self)
        super(ClassNLLCriterionUnstable, self).__init__()

    def updateOutput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)

        self.output = -np.sum(target * np.log(input_clamp)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)

        self.gradInput = -(target / input_clamp) / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterionUnstable"


## 15. (0.3) Negative LogLikelihood criterion (numerically stable)
- input:   **`batch_size x n_feats`** - log probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**

Task is similar to the previous one, but now the criterion input is the output of log-softmax layer. This decomposition allows us to avoid problems with computation of forward and backward of log().

In [ ]:
class ClassNLLCriterion(Criterion):
    def __init__(self):
        a = super(ClassNLLCriterion, self)
        super(ClassNLLCriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = -np.sum(target * input) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput = -target / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterion"


1-я часть задания: реализация слоев, лосей и функций активации - 5 баллов. \\
2-я часть задания: реализация моделей на своих классах. Что должно быть:
  1. Выберите оптимизатор и реализуйте его, чтоб он работал с вами классами. - 1 балл.
  2. Модель для задачи мультирегрессии на выбраных вами данных. Использовать FCNN, dropout, batchnorm, MSE. Пробуйте различные фукнции активации. Для первой модели попробуйте большую, среднюю и маленькую модель. - 1 балл.
  3. Модель для задачи мультиклассификации на MNIST. Использовать свёртки, макспулы, флэттэны, софтмаксы - 1 балла.
  4. Автоэнкодер для выбранных вами данных. Должен быть на свёртках и полносвязных слоях, дропаутах, батчнормах и тд. - 2 балла. \\

Дополнительно в оценке каждой модели будет учитываться:
1. Наличие правильно выбранной метрики и лосс функции.
2. Отрисовка графиков лосей и метрик на трейне-валидации. Проверка качества модели на тесте.
3. Наличие шедулера для lr.
4. Наличие вормапа.
5. Наличие механизма ранней остановки и сохранение лучшей модели.
6. Свитч лося (метрики) и оптимайзера.

> Реализация второй части вынесена в отдельный ноутбук `homework_part2.ipynb`.